# Merging UK and US 2Y Gov Bond Data

In [2]:
import pandas as pd

## Load Data

In [ ]:
# Load dtype map
dtype_map = pd.read_csv("../data/cleaned/2y_gilt/v1_11-09-25/2y_gilt_cleaned_dtypes_as_of_110925.csv", index_col=0).to_dict()

# Convert dtype strings to actual Python types
dtype_map = {
    col: eval(dtype_str) if "float" in dtype_str or "int" in dtype_str else "object"
    for col, dtype_str in dtype_map.items()
}

# Load cleaned data with correct types
uk_2y_clean = pd.read_csv("../data/cleaned/2y_gilt/v1_11-09-25/2y_gilt_cleaned_as_of_110925.csv", dtype=dtype_map, parse_dates=[0])

In [9]:
uk_2y_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12175 entries, 0 to 12174
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    12175 non-null  datetime64[ns]
 1   uk_2y   12175 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 190.4 KB


In [11]:
# Load dtype map
dtype_map = pd.read_csv("../data/cleaned/2y_treasury/v1_11-09-25/2y_treasury_cleaned_dtypes_as_of_110925.csv", index_col=0).to_dict()

# Convert dtype strings to actual Python types
dtype_map = {
    col: eval(dtype_str) if "float" in dtype_str or "int" in dtype_str else "object"
    for col, dtype_str in dtype_map.items()
}

# Load cleaned data with correct types
us_2y_clean = pd.read_csv("../data/cleaned/2y_treasury/v1_11-09-25/2y_treasury_cleaned_as_of_110925.csv", dtype=dtype_map, parse_dates=[0])

In [12]:
us_2y_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12856 entries, 0 to 12855
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    12856 non-null  datetime64[ns]
 1   us_2y   12856 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 201.0 KB


### Indexing

In [ ]:
uk_2y_clean = uk_2y_clean.set_index("date").sort_index()
us_2y_clean = us_2y_clean.set_index("date").sort_index()

In [ ]:
print(uk_2y_clean.info())
print(us_2y_clean.info())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 12175 entries, 1979-01-01 to 2025-08-29
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   uk_2y   12175 non-null  float64
dtypes: float64(1)
memory usage: 190.2 KB
None
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 12856 entries, 1976-06-01 to 2025-09-09
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   us_2y   12856 non-null  float64
dtypes: float64(1)
memory usage: 200.9 KB
None


## Calendar Alignment

In [17]:
start = max(uk_2y_clean.index.min(), us_2y_clean.index.min())
end   = min(uk_2y_clean.index.max(), us_2y_clean.index.max())
calendar = pd.date_range(start=start, end=end, freq="B")

In [18]:
uk_2y_clean = uk_2y_clean.reindex(calendar)
us_2y_clean = us_2y_clean.reindex(calendar)

In [22]:
uk_2y_clean[uk_2y_clean["uk_2y"].isna()]

,uk_2y


In [ ]:
uk_2y_clean[us_2y_clean["us_2y"].isna()]

,us_2y


## Merging

In [27]:
bonds = pd.concat([uk_2y_clean, us_2y_clean], axis=1)

print(bonds.info())
bonds.head()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 12175 entries, 1979-01-01 to 2025-08-29
Freq: B
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   uk_2y   12175 non-null  float64
 1   us_2y   12175 non-null  float64
dtypes: float64(2)
memory usage: 285.4 KB
None


,uk_2y,us_2y
1979-01-01,11.96,9.98
1979-01-02,11.96,10.00
1979-01-03,11.99,9.98
1979-01-04,11.94,9.88
1979-01-05,11.89,9.87


## Export

In [28]:
bonds.to_csv("../data/cleaned/2y_merged/v1_11-09-25/2y_merged_cleaned_as_of_110925.csv")
bonds.dtypes.to_csv("../data/cleaned/2y_merged/v1_11-09-25/2y_merged_cleaned_dtypes_as_of_110925.csv")